In [0]:
from src.ingestion.data_generator import generate_transactions
from pyspark.sql.functions import max as spark_max

# Step 1: Get current max txn_id from Unity Catalog table
try:
    existing_df = spark.table("banking.banking.banking_raw_transactions")
    max_id = existing_df.select(spark_max("txn_id")).collect()[0][0]
    
    if max_id is None:
        max_id = 0
except:
    max_id = 0

print(f"Current Max txn_id: {max_id}")


# Step 2: Generate new data
pdf = generate_transactions(1000)

# Step 3: Offset txn_id to avoid duplicates
pdf["txn_id"] = pdf["txn_id"] + max_id + 1

# Step 4: Convert to Spark DataFrame
df = spark.createDataFrame(pdf)

display(df)


# Step 5: Append to Unity Catalog table (IMPORTANT)
df.write.format("delta") \
    .mode("append") \
    .saveAsTable("banking.banking.banking_raw_transactions")